In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
pip install evaluate sacrebleu rouge_score bert_score

In [1]:
from unsloth import FastModel 
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-10-24 14:39:01.369414: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761316741.392507     645 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761316741.399352     645 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = 2048, 
    load_in_4bit = True,  
    load_in_8bit = False,
    full_finetuning = False,
)

==((====))==  Unsloth 2025.10.9: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Gemma3 does not support SDPA - switching to fast eager.


In [3]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, 
    finetune_language_layers   = True, 
    finetune_attention_modules = True,  
    finetune_mlp_modules       = True,  

    r = 8,           
    lora_alpha = 8,  
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


In [4]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

In [5]:
import pandas as pd 
df = pd.read_csv("/kaggle/input/soap-data/medical_dialogue_train.csv")

train_df = df.sample(frac=0.8, random_state=42)
test_df = df.drop(train_df.index)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [6]:
from datasets import Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [7]:
instruction = "You are an expert medical assistant that generates structured SOAP notes based on the conversation."
def convert_to_conversation(sample):
    conversation = [
        { "role": "user",
          "content" : [
            {"type" : "text",  "text"  : instruction},
            {"type" : "text", "text" : sample["dialogue"]} ]
        },
        { "role" : "assistant",
          "content" : [
            {"type" : "text",  "text"  : sample["soap"]} ]
        },
    ]
    return { "messages" : conversation }
pass

In [8]:
train_dataset = [convert_to_conversation(sample) for sample in train_dataset]
train_dataset[0]

{'messages': [{'role': 'user',
   'content': [{'type': 'text',
     'text': 'You are an expert medical assistant that generates structured SOAP notes based on the conversation.'},
    {'type': 'text',
     'text': "Doctor: Hello! I see that you're a 73-year-old female with a past medical history of hypertension, hyperlipidemia, insulin-dependent diabetes mellitus, and coronary artery disease. You were admitted for an elective percutaneous coronary intervention (PCI) of a lesion detected in the right coronary artery. Can you tell me more about that and any symptoms you've been experiencing?\nPatient: Yes, I had old stents in my mid-right coronary artery and mid-left anterior descending artery, and a stent in the mid-left circumflex artery that was put in three weeks before admission.\nDoctor: Alright. On the day of the planned PCI, shortly after right femoral artery cannulation, you developed a sudden complete heart block. We immediately administered Atropine and inserted a temporary pa

In [9]:
test_dataset = [convert_to_conversation(sample) for sample in test_dataset]
test_dataset[0]

{'messages': [{'role': 'user',
   'content': [{'type': 'text',
     'text': 'You are an expert medical assistant that generates structured SOAP notes based on the conversation.'},
    {'type': 'text',
     'text': "Doctor: Hello, what brings you in today?\nPatient: Hi, my 21-month-old son has been experiencing weakness in his lower extremities and lumbar pain after a mild upper respiratory tract infection.\nDoctor: I see. Did you consult any medical professional for this issue?\nPatient: Yes, we took him to the pediatric emergency department. They did a hip ultrasound, but they didn't find any joint effusion. They dismissed it as transient hips arthritis and treated him with anti-inflammatory therapy.\nDoctor: Did the therapy help with his condition?\nPatient: Unfortunately, no. His pain got worse, and now he's unable to walk. That's why we came back here.\nDoctor: We should definitely investigate further. We'll start by admitting him to the hospital. We'll need to perform a spine radi

In [10]:
from unsloth.chat_templates import standardize_data_formats
train_dataset = standardize_data_formats(train_dataset)
test_dataset = standardize_data_formats(test_dataset)

In [11]:
train_dataset = Dataset.from_list(train_dataset)
test_dataset = Dataset.from_list(test_dataset)
def formatting_prompts_func(examples):
   convos = examples["messages"]
   texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
   return { "text" : texts, }

train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
test_dataset = test_dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/7400 [00:00<?, ? examples/s]

Map:   0%|          | 0/1850 [00:00<?, ? examples/s]

In [12]:
messages = [{
    "role": "user",
    "content": [{
        "type" : "text",
        "text" : train_dataset[0]
    },{
        "type" : "text",
        "text" : instruction,
    }]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
)

from transformers import TextStreamer
generated = model.generate(
    **inputs.to("cuda"),
    max_new_tokens = 512, 
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Okay, here’s the structured SOAP note based on the provided conversation:

**S: Subjective**

*   **Age/Sex:** 73-year-old female
*   **History:** Hypertension, hyperlipidemia, insulin-dependent diabetes mellitus, coronary artery disease.
*   **Chief Complaint:** Elective PCI for lesion in right coronary artery.
*   **Past Medical History:** Previous stents in mid-right coronary artery, mid-left anterior descending artery, and mid-left circumflex artery (placed 3 weeks prior).
*   **Recent Events:**  Reports experiencing a sudden complete heart block, SVT with aberrancy, acute pulmonary edema, hypotension, and hypoxia during the PCI procedure.  Patient describes the experience as “scary” and “tough.”  Patient recalls receiving IV adenosine, transitioning to sinus tachycardia.



**O: Objective**

*   **Initial Findings:**  Complete heart block requiring Atropine and temporary pacemaker insertion. Subsequent SVT with aberrancy.
*   **Intervention:**  IV adenosine administered. Intubatio

In [13]:
import evaluate
from bert_score import score as bert_score
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
reference_text = train_df["soap"].iloc[0]

gen_tokens = generated[0, inputs["input_ids"].shape[-1]:]
generated_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

rouge_result = rouge.compute(predictions=[generated_text.strip()], references=[reference_text.strip()], use_stemmer=True)
rouge_scores = rouge_result
bleu_result = bleu.compute(
        predictions= [generated_text.strip()],
        references=[[ref] for ref in [reference_text.strip()]]  
    )
P, R, F1 = bert_score([generated_text.strip()], [reference_text.strip()], lang="en", verbose=True)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.50 seconds, 1.99 sentences/sec


In [14]:
print("___Before Fine Tuning___")
print(f"ROUGE-1: {rouge_scores['rouge1']}")
print(f"ROUGE-2: {rouge_scores['rouge2']}")
print(f"ROUGE-L: {rouge_scores['rougeL']}")
print(f"BLEU: {bleu_result['bleu']}")
print(f"BERTScore: {F1.mean().item()}")

___Before Fine Tuning___
ROUGE-1: 0.7047619047619048
ROUGE-2: 0.4435946462715105
ROUGE-L: 0.5866666666666667
BLEU: 0.24216645172173215
BERTScore: 0.8693757653236389


In [15]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = test_dataset, 
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, 
        warmup_steps = 5,
        #num_train_epochs = 1, 
        max_steps = 100,
        learning_rate = 2e-4, 
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", 
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/7400 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/1850 [00:00<?, ? examples/s]

In [16]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

Map (num_proc=8):   0%|          | 0/7400 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/1850 [00:00<?, ? examples/s]

In [17]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,400 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 14,901,248 of 4,314,980,720 (0.35% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.886600
2,1.962000
3,2.003800
4,1.662500
5,1.394100
6,1.154200
7,0.977100
8,1.011300
9,1.027200
10,0.861500


In [18]:
messages = [{
    "role": "user",
    "content": [{
        "type" : "text",
        "text" : train_dataset[0]
    },{
        "type" : "text",
        "text" : instruction,
    }]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
)

from transformers import TextStreamer
generated = model.generate(
    **inputs.to("cuda"),
    max_new_tokens = 512, 
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

S: 73-year-old female presented for elective PCI of the right coronary artery due to an old lesion. Patient reports previous interventions for mid-right, mid-left anterior descending, and mid-left circumflex coronary artery lesions. The critical event during PCI was sudden Complete Heart Block with subsequent SVT with aberrancy, presenting as a "scary experience" as described by the patient.
O: On the event, the patient experienced a rapid shift from Complete Heart Block and SVT to sinus tachycardia, acute pulmonary edema with associated hypotension and hypoxia. Intervention included IV Adenosine, temporary pacemaker, mechanical ventilation, and intra-aortic balloon pump (IABP). Subsequent echosho indicated severe left ventricular dyskinesis (EF < 20% with resolution to 60% EF post-acute event, with complete resolution of apical wall motion abnormalities. No changes noted from previous angiogram.
A: Primary Diagnosis was Takotsubo Syndrome following PCI. Differential diagnosis initiall

In [19]:
import evaluate
from bert_score import score as bert_score
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
reference_text = train_df["soap"].iloc[0]

gen_tokens = generated[0, inputs["input_ids"].shape[-1]:]
generated_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

rouge_result = rouge.compute(predictions=[generated_text.strip()], references=[reference_text.strip()], use_stemmer=True)
rouge_scores = rouge_result
bleu_result = bleu.compute(
        predictions= [generated_text.strip()],
        references=[[ref] for ref in [reference_text.strip()]]  
    )
P, R, F1 = bert_score([generated_text.strip()], [reference_text.strip()], lang="en", verbose=True)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.21 seconds, 4.83 sentences/sec


In [20]:
print("___After Fine Tuning___")
print(f"ROUGE-1: {rouge_scores['rouge1']}")
print(f"ROUGE-2: {rouge_scores['rouge2']}")
print(f"ROUGE-L: {rouge_scores['rougeL']}")
print(f"BLEU: {bleu_result['bleu']}")
print(f"BERTScore: {F1.mean().item()}")

___After Fine Tuning___
ROUGE-1: 0.6315789473684211
ROUGE-2: 0.28048780487804875
ROUGE-L: 0.3684210526315789
BLEU: 0.18532722438342175
BERTScore: 0.895370602607727


In [21]:
model.save_pretrained("UIU_AIMScribe_gemma-3_model") 
tokenizer.save_pretrained("UIU_AIMScribe_gemma-3_tokenizer")

['UIU_AIMScribe_gemma-3_tokenizer/processor_config.json']

In [ ]:
# model.push_to_hub("HF_ACCOUNT/gemma-3", token = "...") # Online saving
# tokenizer.push_to_hub("HF_ACCOUNT/gemma-3", token = "...") # Online saving